In [9]:
%cd '/Users/lorenzodelcol/Desktop/RA/REPLICATION DROPBOX (NEW)/Raw data UPDATE/Updated 2026 Series'

/Users/lorenzodelcol/Desktop/RA/REPLICATION DROPBOX (NEW)/Raw data UPDATE/Updated 2026 Series


/opt/anaconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [10]:
import pandas as pd

In [12]:
print("Building Phase C: Auto-filling the GFU Dataset...")

# 1. Load the "Skeleton" (The original GFU file) to copy its exact column structure
gfu_skeleton = pd.read_excel('cc_globalfactor_1992M72020M5.xlsx', sheet_name='raw data')
target_columns = gfu_skeleton.columns.tolist()

# 2. Load the three Phase B dataframes
# Setting 'Date' as the index and parsing dates makes time-series alignment automatic
mkt_df = pd.read_excel('PHASE_B_MKT_monthly_df.xlsx', index_col='Date', parse_dates=True)
ex_df  = pd.read_excel('PHASE_B_EX_monthly_df.xlsx', index_col='Date', parse_dates=True)
bond_df = pd.read_excel('PHASE_B_10YR_monthly_df.xlsx', index_col='Date', parse_dates=True)

# 3. Create the New Timeline
# Start from July 1992 and end at the maximum date found in any of your Phase B files
start_date = '1992-07-01'
max_date = max(mkt_df.index.max(), ex_df.index.max(), bond_df.index.max())
new_dates = pd.date_range(start=start_date, end=max_date, freq='MS') # 'MS' = Month Start

# 4. Create the blank Master DataFrame
new_gfu = pd.DataFrame(index=new_dates, columns=target_columns)
new_gfu = new_gfu.drop(columns=['Date']) # Drop 'Date' temporarily as it is currently our index

# 5. The Mapping Engine
for col in new_gfu.columns:
    # GFU columns look like 'US_stock' or 'Germany_BY'. 
    # We split from the right side ('_stc', '_EX', '_BY') to isolate the country name and the variable type.
    parts = col.rsplit('_', 1)
    
    if len(parts) == 2:
        # Replaces underscores in country names (e.g., 'South_Africa' -> 'South Africa')
        country = parts[0].replace('_', ' ') 
        var_type = parts[1].lower()
        
        # This is the target column we are looking for in your Phase B files
        sd_col_name = f"SD {country}"
        
        try:
            # Route Stocks -> MKT DataFrame
            if var_type.startswith('st') or var_type == 's': 
                if sd_col_name in mkt_df.columns:
                    new_gfu[col] = mkt_df[sd_col_name]
                    
            # Route Exchange Rates -> EX DataFrame
            elif var_type.startswith('ex') or var_type == 'e': 
                if sd_col_name in ex_df.columns:
                    new_gfu[col] = ex_df[sd_col_name]
                    
            # Route Bond Yields -> 10YR DataFrame
            elif var_type.startswith('by') or var_type == 'b': 
                if sd_col_name in bond_df.columns:
                    new_gfu[col] = bond_df[sd_col_name]
                    
        except Exception as e:
            print(f"Warning: Could not process column {col}. Error: {e}")

# 6. Final Polish
new_gfu.index.name = 'Date'
new_gfu.reset_index(inplace=True)

# Format the dates cleanly as DD/MM/YYYY
new_gfu['Date'] = pd.to_datetime(new_gfu['Date']).dt.strftime('%d/%m/%Y')

# 7. Save the final output
output_name = 'PHASE_C_Updated_GFU_Dataset.xlsx'
new_gfu.to_excel(output_name, index=False)

print(f"Success! Your new dataset is ready and saved as '{output_name}'")

Building Phase C: Auto-filling the GFU Dataset...


/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/var/folders/_7/z6fx6jdx1h90fmhzn2dtnxfr0000gn/T/ipykernel_22401/11077658.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_gfu.reset_index(inplace=True)


Success! Your new dataset is ready and saved as 'PHASE_C_Updated_GFU_Dataset.xlsx'
